# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đườngĐọc `README.md` và `GUIDE.md` trước. Notebook này cài đặt theo pseudo code trong GUIDE.

In [ ]:
# Runtime -> Change runtime type -> GPUimport osfrom pathlib import Pathif not Path("scripts/labcommon.py").exists():    !git clone https://github.com/nghon4maeri/K4-Track-4-Day10-VisionLanguageModel.git    %cd K4-Track-4-Day10-VisionLanguageModel!pip -q install -r requirements-colab.txt!nvidia-smi -L

In [ ]:
import ostry:    from google.colab import userdata    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")except Exception:    pass  # Kaggle injects credentials automatically

## 0. Môi trường (Colab T4)

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
from pathlib import Pathimport json, randomROOT = Path('.')meta = json.load(open('data/splits/meta.json', encoding='utf-8'))print('dataset_version:', meta['dataset_version'])for sp in ['train','val','test']:    rows=[json.loads(l) for l in open(f'data/splits/{sp}.jsonl', encoding='utf-8')]    from collections import Counter    c=Counter()    for r in rows:        for o in r['objects']:            c[o['label']]+=1    print(sp, 'images=', len(rows), dict(c))

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
from pathlib import Pathimport jsonfrom collections import Counterfor sp in ['train','val','test']:    rows=[json.loads(l) for l in open(f'data/splits/{sp}.jsonl', encoding='utf-8')]    c=Counter(o['label'] for r in rows for o in r['objects'])    print(sp, len(rows), dict(c))

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import jsonCLASSES = ['pothole','crack','manhole']def yolo_to_box(label, yolo):    xc, yc, w, h = yolo    x1 = (xc - w/2) * 1000; y1 = (yc - h/2) * 1000    x2 = (xc + w/2) * 1000; y2 = (yc + h/2) * 1000    def clamp(v): return int(min(1000, max(0, round(v))))    return {'label': label, 'bbox_2d': [clamp(x1), clamp(y1), clamp(x2), clamp(y2)]}def objects_to_answer(objects):    boxes = [yolo_to_box(o['label'], o['yolo']) for o in objects]    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))    return json.dumps(boxes, separators=(',', ':'))def objects_to_answer_px(objects, width=640, height=360):    boxes = []    for o in objects:        xc, yc, w, h = o['yolo']        x1 = (xc - w/2) * width;  y1 = (yc - h/2) * height        x2 = (xc + w/2) * width;  y2 = (yc + h/2) * height        def clamp(v, mx): return int(min(mx, max(0, round(v))))        boxes.append({'label': o['label'], 'bbox_2d': [clamp(x1,width), clamp(y1,height), clamp(x2,width), clamp(y2,height)]})    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))    return json.dumps(boxes, separators=(',', ':'))PROMPT = ('You are a road damage detector. Return a JSON list of objects with keys '          '"label" and "bbox_2d". Labels are one of: pothole, crack, manhole. '          'Coordinates are integers on the 0-1000 scale relative to image width/height. '          'Return [] if none. Output only the JSON, no explanation.')PROMPT_PIXEL = ('You are a road damage detector. Return a JSON list of objects with keys '                '"label" and "bbox_2d". Labels are one of: pothole, crack, manhole. '                'Coordinates are integer pixels in the original 640x360 image. '                'Return [] if none. Output only the JSON, no explanation.')print(objects_to_answer(json.load(open('data/splits/val.jsonl',encoding='utf-8'))[0]['objects']))

## 3. E0: zero-shot

In [ ]:
import os, json, timefrom pathlib import Pathimport torchfrom PIL import Imagefrom transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfigMODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',      bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,      llm_int8_skip_modules=['visual','lm_head'])def load_base():    model = Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=BNB,            dtype=torch.float16, device_map={'':0})    proc = AutoProcessor.from_pretrained(MODEL_ID); proc.tokenizer.padding_side='left'    return model, procdef infer_once(model, proc, image, prompt, max_new_tokens=256):    messages=[{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]    text = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)    inputs = proc(text=[text], images=[image], return_tensors='pt')    inputs = {k:v.to('cuda') for k,v in inputs.items()}    with torch.no_grad():        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)    return proc.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0]def run_inference(name, adapter_dir=None, projector_path=None, prompt=PROMPT, coord_scale='norm1000', split='test', out_path=None):    model, proc = load_base()    if adapter_dir is not None:        from peft import PeftModel        model = PeftModel.from_pretrained(model, adapter_dir)    if projector_path is not None and Path(projector_path).exists():        extra = torch.load(projector_path, map_location='cpu')        for n,p in model.named_parameters():            if n in extra:                p.data.copy_(extra[n].to(device=p.device, dtype=p.dtype))    model.eval()    entries = [json.loads(l) for l in open(f'data/splits/{split}.jsonl', encoding='utf-8')]    outs=[]    for i,e in enumerate(entries):        img = Image.open(Path('data')/e['image_path']).convert('RGB')        raw = infer_once(model, proc, img, prompt)        outs.append({'image_id': e['image_id'], 'raw': raw})        if i%50==0: print(name, i, raw[:60])    out_path = out_path or f'work/{name.lower()}.jsonl'    Path(out_path).parent.mkdir(parents=True, exist_ok=True)    from scripts.labcommon import write_jsonl    write_jsonl(out_path, outs)    print('wrote', out_path, len(outs))    del model; torch.cuda.empty_cache()    return out_pathMODEL, PROC = load_base()# E0entries = [json.loads(l) for l in open('data/splits/test.jsonl', encoding='utf-8')]outs=[]for i,e in enumerate(entries):    img = Image.open(Path('data')/e['image_path']).convert('RGB')    raw = infer_once(MODEL, PROC, img, PROMPT)    outs.append({'image_id': e['image_id'], 'raw': raw})    if i%50==0: print('E0', i, raw[:60])from scripts.labcommon import write_jsonlwrite_jsonl('work/e0.jsonl', outs)del MODEL; torch.cuda.empty_cache()print('E0 done', len(outs))

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
import torchdef encode_example(image, prompt, answer, processor):    user_msg = {'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}    asst_msg = {'role':'assistant','content':[{'type':'text','text':answer}]}    prompt_enc = processor.apply_chat_template([user_msg], images=[image], tokenize=True,                   add_generation_prompt=True, return_dict=True, return_tensors='pt')    full_enc = processor.apply_chat_template([user_msg, asst_msg], images=[image], tokenize=True,                   add_generation_prompt=False, return_dict=True, return_tensors='pt')    n = prompt_enc['input_ids'].shape[1]    # kiem tra prompt la tien to    assert torch.equal(full_enc['input_ids'][0,:n], prompt_enc['input_ids'][0]), 'prompt khong phai tien to'    labels = full_enc['input_ids'].clone()    labels[:, :n] = -100    out = dict(full_enc)    out['labels'] = labels    # an token anh    enc_img_tokens = (out['input_ids']==processor.tokenizer.additional_special_tokens_ids[...]) if False else None    return out, prompt_enc# test nhanh 1 vi duimport jsonfrom PIL import Imagee = [json.loads(l) for l in open('data/splits/train.jsonl',encoding='utf-8')][0]img = Image.open(Path('data')/e['image_path']).convert('RGB') if False else Nonefrom pathlib import Pathimg = Image.open(Path('data')/e['image_path']).convert('RGB')ans = objects_to_answer(e['objects'])enc, pe = encode_example(img, PROMPT, ans, PROC if 'PROC' in globals() else (load_base()[1]))print('seq len', enc['input_ids'].shape[1])print('image tokens ~', int((enc['mm_token_type_ids']==1).sum()) if 'mm_token_type_ids' in enc else 'NA')# giai ma token labels != -100ids = enc['input_ids'][0][enc['labels'][0]!=-100]# can processor tu loadmodel_tmp, proc_tmp = load_base()print('answer decode:', proc_tmp.tokenizer.decode(ids, skip_special_tokens=True)[:200])del model_tmp

In [ ]:
def collate(batch):    out = {}    for k in ['input_ids','attention_mask','mm_token_type_ids','labels']:        if k not in batch[0]: continue        maxlen = max(b[k].shape[-1] for b in batch)        pad_val = -100 if k=='labels' else 0        tensors=[]        for b in batch:            t = b[k]            if t.dim()==1: t=t.unsqueeze(0)            pad = maxlen - t.shape[-1]            tensors.append(torch.nn.functional.pad(t, (0,pad), value=pad_val))        out[k] = torch.cat(tensors, dim=0)    for k in ['pixel_values','image_grid_thw']:        if k in batch[0]:            out[k] = torch.cat([b[k] for b in batch], dim=0)    return out

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
import refrom peft import LoraConfig, get_peft_modeldef apply_setup(model, setup, rank=16, alpha=32):    for p in model.parameters():        p.requires_grad_(False)    model.enable_input_require_grads()    patterns = [r'language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)']    vit = (setup == 'E3')    projector = setup in ('E2','E3')    if vit:        patterns.append(r'visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2))')    combined = '|'.join(f'({p})' for p in patterns)    cfg = LoraConfig(r=rank, lora_alpha=alpha, lora_dropout=0.05,                     target_modules=combined, bias='none', task_type='CAUSAL_LM')    model = get_peft_model(model, cfg)    if projector:        for n,p in model.named_parameters():            if 'visual.merger' in n or 'visual.deepstack_merger_list' in n:                p.data = p.data.to(torch.float32)                p.requires_grad_(True)    matched = [n for n,_ in model.named_modules() if re.search(combined, n)]    print(setup, 'matched_modules=', len(matched), 'trainable_params=', sum(p.numel() for p in model.parameters() if p.requires_grad))    return model

## 6. Vòng huấn luyện

In [ ]:
from PIL import Image
import os, random, time, jsonimport torchfrom torch.utils.data import Dataset, DataLoaderfrom torch.optim import AdamWfrom torch.cuda.amp import GradScalerfrom transformers import get_cosine_schedule_with_warmupfrom pathlib import Pathclass VLDataset(Dataset):    def __init__(self, entries, prompt, answer_fn, processor, fraction=1.0, seed=42):        rng = random.Random(seed)        idx = list(range(len(entries)))        rng.shuffle(idx)        k = max(1, int(len(entries)*fraction)) if fraction<1 else len(entries)        self.sel = [entries[i] for i in idx[:k]]        self.prompt = prompt; self.answer_fn = answer_fn        self.processor = processor    def __len__(self): return len(self.sel)    def __getitem__(self, i):        e = self.sel[i]        img = Image.open(Path('data')/e['image_path']).convert('RGB')        ans = self.answer_fn(e['objects'])        enc, _ = encode_example(img, self.prompt, ans, self.processor)        return encdef train_run(setup, name, rank=16, alpha=32, train_fraction=1.0, coord_scale='norm1000', max_steps=300):    prompt = PROMPT if coord_scale=='norm1000' else PROMPT_PIXEL    ans_fn = (lambda objs: objects_to_answer(objs)) if coord_scale=='norm1000' else (lambda objs: objects_to_answer_px(objs))    model, processor = load_base()    model = apply_setup(model, setup, rank, alpha)    entries = [json.loads(l) for l in open('data/splits/train.jsonl', encoding='utf-8')]    ds = VLDataset(entries, prompt, ans_fn, processor, fraction=train_fraction, seed=42)    loader = DataLoader(ds, batch_size=2, shuffle=True, collate_fn=collate)    trainable = [p for p in model.parameters() if p.requires_grad]    trainable_params = sum(p.numel() for p in trainable)    opt = AdamW(trainable, lr=2e-4, weight_decay=0)    warmup = max(1, int(0.05*max_steps))    sched = get_cosine_schedule_with_warmup(opt, warmup, max_steps)    scaler = GradScaler('cuda')    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})    model.config.use_cache = False    model.train()    step = 0; t0 = time.time()    peak = 0    opt.zero_grad()    while step < max_steps:        accum = 0        for batch in loader:            if step >= max_steps: break            batch = {k:(v.to('cuda') if torch.is_tensor(v) else v) for k,v in batch.items()}            with torch.autocast('cuda', dtype=torch.float16):                out = model(**batch)                loss = out.loss / 4            if not torch.isfinite(loss):                raise RuntimeError('non-finite loss')            scaler.scale(loss).backward()            accum += 1            if accum == 4:                scaler.unscale_(opt)                torch.nn.utils.clip_grad_norm_(trainable, 1.0)                scaler.step(opt); scaler.update(); opt.zero_grad()                sched.step()                step += 1                peak = max(peak, torch.cuda.max_memory_allocated())                if step % 100 == 0 or step == max_steps:                    ckpt = f'work/{name}_step{step}'                    os.makedirs(ckpt, exist_ok=True)                    model.save_pretrained(ckpt)                    extra = {n:p.detach().cpu() for n,p in model.named_parameters()                             if p.requires_grad and ('visual.merger' in n or 'visual.deepstack_merger_list' in n)}                    torch.save(extra, f'{ckpt}/projector.pt')                    print('saved', ckpt, 'step', step)                accum = 0        if accum != 0:            opt.zero_grad()    adapter_dir = f'work/{name}_adapter'    os.makedirs(adapter_dir, exist_ok=True)    model.save_pretrained(adapter_dir)    extra = {n:p.detach().cpu() for n,p in model.named_parameters()             if p.requires_grad and ('visual.merger' in n or 'visual.deepstack_merger_list' in n)}    torch.save(extra, f'work/{name}_projector.pt')    minutes = (time.time()-t0)/60    run = {'trainable_params': int(trainable_params), 'peak_vram_gb': round(peak/1e9,3),           'train_minutes': round(minutes,2)}    json.dump(run, open(f'work/{name}_run.json','w'))    print(name, run)    del model; torch.cuda.empty_cache()    return adapter_dir, f'work/{name}_projector.pt', run

## 7. Suy luận và chấm E1–E3

In [ ]:
# E1a1,p1,r1 = train_run('E1','E1')run_inference('E1', adapter_dir=a1, projector_path=None, out_path='work/e1.jsonl')json.dump if False else None

In [ ]:
!python scripts/score.py --pred work/e1.jsonl --name E1 --out report/evidence/e1.json --meta trainable_params=$(python -c "import json;print(json.load(open('work/E1_run.json'))['trainable_params'])") --meta peak_vram_gb=$(python -c "import json;print(json.load(open('work/E1_run.json'))['peak_vram_gb'])") --meta train_minutes=$(python -c "import json;print(json.load(open('work/E1_run.json'))['train_minutes'])")

In [ ]:
# E2a2,p2,r2 = train_run('E2','E2')run_inference('E2', adapter_dir=a2, projector_path=p2, out_path='work/e2.jsonl')

In [ ]:
!python scripts/score.py --pred work/e2.jsonl --name E2 --out report/evidence/e2.json --meta trainable_params=$(python -c "import json;print(json.load(open('work/E2_run.json'))['trainable_params'])") --meta peak_vram_gb=$(python -c "import json;print(json.load(open('work/E2_run.json'))['peak_vram_gb'])") --meta train_minutes=$(python -c "import json;print(json.load(open('work/E2_run.json'))['train_minutes'])")

In [ ]:
# E3a3,p3,r3 = train_run('E3','E3')run_inference('E3', adapter_dir=a3, projector_path=p3, out_path='work/e3.jsonl')

In [ ]:
!python scripts/score.py --pred work/e3.jsonl --name E3 --out report/evidence/e3.json --meta trainable_params=$(python -c "import json;print(json.load(open('work/E3_run.json'))['trainable_params'])") --meta peak_vram_gb=$(python -c "import json;print(json.load(open('work/E3_run.json'))['peak_vram_gb'])") --meta train_minutes=$(python -c "import json;print(json.load(open('work/E3_run.json'))['train_minutes'])")

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4 ablation: rank (chon setup tot nhat, vi du E2)a_lo,p_lo,_ = train_run('E2','E4_rank_4', rank=4, alpha=8)a_hi,p_hi,_ = train_run('E2','E4_rank_64', rank=64, alpha=128)run_inference('E4_r4', adapter_dir=a_lo, projector_path=p_lo, out_path='work/e4_rank_4.jsonl')run_inference('E4_r64', adapter_dir=a_hi, projector_path=p_hi, out_path='work/e4_rank_64.jsonl')

In [ ]:
!python scripts/score.py --pred work/e4_rank_4.jsonl --name E4_rank_4 --out report/evidence/e4_rank_4.json!python scripts/score.py --pred work/e4_rank_64.jsonl --name E4_rank_64 --out report/evidence/e4_rank_64.json

In [ ]:
# E5: reward IoU (Python thuan, co test)import json, refrom scripts.score import parse_output, iou_xyxydef reward_iou(completion, gt_boxes):    # gt_boxes: list [{'label':..., 'bbox':(x1,y1,x2,y2)}]    pred = parse_output(completion)    if pred is None: return 0.0    if not gt_boxes and not pred: return 1.0    # ghep doi tham lam theo IoU giam dan, cung lop, IoU>=0.5    pairs = []    for gi,g in enumerate(gt_boxes):        for pi,p in enumerate(pred):            if g['label']==p['label']:                v = iou_xyxy(g['bbox'], p['bbox'])                if v>=0.5:                    pairs.append((v,gi,pi))    pairs.sort(reverse=True)    used_g=set(); used_p=set(); total_iou=0.0    for v,gi,pi in pairs:        if gi in used_g or pi in used_p: continue        used_g.add(gi); used_p.add(pi); total_iou+=v    return 0.2 + 0.8*total_iou/max(len(gt_boxes), len(pred), 1)# testsgt=[{'label':'pothole','bbox':(0,0,10,10)}]assert reward_iou('[{"label":"pothole","bbox_2d":[0,0,10,10]}]', gt) > 0.9assert reward_iou('not json', gt)==0.0assert reward_iou('[]', gt)==0.2assert reward_iou('[]', [])==1.0print('reward tests ok')